# Boccia Data Analysis 

## Import Libraries

In [1]:
import numpy as np
import pyxdf
import os

## Import LSL Streams

In [2]:
file_path = "D:/Daniella Bourque/3-28_Testing_Data" # Path to the folder containing the data
filename = os.path.join(file_path, "sub-P001_ses-S001_task-Default_run-001_eeg.xdf")

streams, fileheader = pyxdf.load_xdf(filename)

for [s, stream] in enumerate(streams):
    print(f"Stream {s}: {stream['info']['name'][0]}")

Stream 0: PythonResponse
Stream 1: TargetElementStream_VirtualPlay
Stream 2: UnityMarkerStream
Stream 3: DSI7-flex


## Actual Target Elements

In [3]:
# Select target element stream
target_stream_index = next((i for i, stream in enumerate(streams) if "TargetElementStream_VirtualPlay" in stream['info']['name'][0]), None)
print(f"Stream {target_stream_index}: {streams[target_stream_index]['info']['name'][0]}")
target_element_stream = streams[target_stream_index]
target_element_stream_markers = target_element_stream['time_series']
target_element_stream_time = target_element_stream['time_stamps']

Stream 1: TargetElementStream_VirtualPlay


In [4]:
# Extract Selectable Object indices (iSPOs) from target element stream
target_element_iSPOs = [int(item[1].split(": ")[1]) for item in target_element_stream_markers]
print(target_element_iSPOs)

[13, 14, 12, 2, 29, 26, 4, 13, 1, 5, 3]


## Bessy Python Predictions: from Python response stream

In [5]:
# Select Python response stream
python_response_stream_index = next((i for i, stream in enumerate(streams) if "PythonResponse" in stream['info']['name'][0]), None)
print(f"Stream {python_response_stream_index}: {streams[python_response_stream_index]['info']['name'][0]}")

python_response_stream = streams[python_response_stream_index]
python_response_stream_markers = python_response_stream['time_series']

Stream 0: PythonResponse


In [6]:
# Extract predictions from Python response stream by removing 'ping' and 'marker received' messages
extracted_predictions = [
    s for sublist in python_response_stream_markers for s in sublist
    if 'ping' not in str(s).lower() and 'marker received' not in str(s).lower()
]

predicted_targets = [int(s.strip('[]')) for s in extracted_predictions] # Format as list of integers
print(predicted_targets)

[13, 10, 5, 2, 29, 26, 10, 13, 1, 6, 3]


## Compare predicted targets to actual targets

In [7]:
# Make sure number of predictions matches number of actual targets
assert len(predicted_targets) == len(target_element_iSPOs)

In [8]:
# Determine number of correct predictions
num_correct = 0
num_total = len(predicted_targets)
for i in range(num_total):
    if predicted_targets[i] == target_element_iSPOs[i]:
        num_correct += 1

In [9]:
# Calculate accuracy
accuracy = num_correct / num_total

print(f"Number of correct predictions: {num_correct} correct out of {num_total} total.")
print(f"Accuracy: {accuracy}")

Number of correct predictions: 7 correct out of 11 total.
Accuracy: 0.6363636363636364
